## Generating mosaics by assembling all the NDVI and NDWI images, respectively

### Libraries and general paths

In [ ]:
!pip install rasterio
!pip install geopandas

In [ ]:
import glob
import rasterio
from rasterio.transform import from_origin
import numpy as np
from tqdm import tqdm
import os
import geopandas as gpd
from math import ceil
import matplotlib.pyplot as plt

Note: Previously to this repository, the ortophotos from the airbone survey were used to compute and classify NDVI and NDWI indexes in the private repository from de Bologna Digital Twin project. In this notebook, we start with those products.  

In [ ]:
ndvi_paths = glob.glob('../../../datos-notebooks/3.HR-data-bologna/3.1.land-cover/temp_ndvi-2/*.tif') # folder with NDVI files
ndwi_paths = glob.glob('../../../datos-notebooks/3.HR-data-bologna/3.1.land-cover/temp_ndwi/*.tif') # folder with NDWI files
bound = '../../../datos-notebooks/3.HR-data-bologna/3.0.base-data/vectors/bologne_boundary.geojson'

output_folder = '../../../datos-notebooks/3.HR-data-bologna/3.1.land-cover/mosaics/'
os.makedirs(output_folder, exist_ok=True)
output_path_ndwi = os.path.join(output_folder, "ndwi_mosaic.tif")
output_path_ndvi = os.path.join(output_folder, "ndvi_mosaic-2.tif")

### Generate one NDWI mosaic with all the NDWI files

In [ ]:
# Parameters 
batch_size = 50  # number of NDWI files per batch

# get mosaic extent from boundary
boundary = gpd.read_file(bound)
minx, miny, maxx, maxy = boundary.total_bounds

# get resolution and nodata from first image
with rasterio.open(ndwi_paths[0]) as src:
    res_x = src.transform.a
    res_y = -src.transform.e
    dtype = src.dtypes[0]
    nodata = src.nodata
    profile_base = src.profile.copy()

# compute mosaic size and transform 
width = ceil((maxx - minx) / res_x)
height = ceil((maxy - miny) / res_y)
transform = from_origin(minx, maxy, res_x, res_y)

# create output mosaic on disk
profile_base.update({
    'height': height,
    'width': width,
    'transform': transform,
    'dtype': 'float32',
    'count': 1,
    'compress': 'lzw',
    'nodata': np.nan
})

# Split NDWI files into batches to avoid memory issues
ndwi_batches = [ndwi_paths[i:i + batch_size] for i in range(0, len(ndwi_paths), batch_size)]

with rasterio.open(output_path_ndwi, "w+", **profile_base) as dst: # open the reultant tiff
    for batch_idx, batch in enumerate(ndwi_batches, 1):
        print(f"Processing batch {batch_idx}/{len(ndwi_batches)}...")
        for ndwi_file in tqdm(batch, desc=f"Batch {batch_idx}"):
            with rasterio.open(ndwi_file) as src:
                data = src.read(1).astype(np.float32)
                rows, cols = data.shape

                # compute offsets relative to mosaic origin
                col_off = int(round((src.bounds.left - minx) / res_x))
                row_off = int(round((maxy - src.bounds.top) / res_y))

                # clip window to mosaic bounds
                win_col_start = max(col_off, 0)
                win_row_start = max(row_off, 0)
                win_col_end = min(col_off + cols, width)
                win_row_end = min(row_off + rows, height)

                # skip tiles completely outside mosaic
                if win_col_start >= win_col_end or win_row_start >= win_row_end:
                    continue

                # corresponding slice of NDWI tile
                data_row_start = max(0, 0 - row_off)
                data_col_start = max(0, 0 - col_off)
                data_row_end = data_row_start + (win_row_end - win_row_start)
                data_col_end = data_col_start + (win_col_end - win_col_start)
                data_slice = (slice(data_row_start, data_row_end), slice(data_col_start, data_col_end))

                # read mosaic block
                mosaic_block = dst.read(1, window=rasterio.windows.Window(
                    win_col_start, win_row_start,
                    win_col_end - win_col_start, win_row_end - win_row_start),
                    boundless=True, fill_value=np.nan
                )

                # fill only where mosaic is nodata (because ortophotos have overlapping)
                tile_block = data[data_slice]
                fill_mask = np.isnan(mosaic_block) & (~np.isnan(tile_block))
                mosaic_block[fill_mask] = tile_block[fill_mask]

                # write back updated block
                dst.write(mosaic_block, 1, window=rasterio.windows.Window(
                    win_col_start, win_row_start,
                    win_col_end - win_col_start, win_row_end - win_row_start)
                )

print(f"NDWI mosaic saved at {output_path_ndwi}")

In [ ]:
# Print a window to check the outputs
with rasterio.open(output_path_ndwi) as src:
    window = rasterio.windows.Window(col_off=10000, row_off=2000, width=8000, height=4000) 
    data = src.read(1, window=window)    
    transform = src.window_transform(window) # get transform for plotting

fig, ax = plt.subplots(figsize=(10,4))
img = ax.imshow(data, cmap='Blues', vmin=0, vmax=1)
ax.set_title("NDWI Mosaic (sample window)")
plt.colorbar(img, ax=ax, label="NDWI")
plt.show()

### Generate one NDVI mosaic with all the NDVI files

In [ ]:
# Parameters  
batch_size = 50  # number of NDVI files per batch

# get mosaic extent from boundary
boundary = gpd.read_file(bound)
minx, miny, maxx, maxy = boundary.total_bounds

# get resolution and nodata from first image
with rasterio.open(ndvi_paths[0]) as src:
    res_x = src.transform.a
    res_y = -src.transform.e
    dtype = src.dtypes[0]
    nodata = src.nodata
    profile_base = src.profile.copy()

# compute mosaic size and transform 
width = ceil((maxx - minx) / res_x)
height = ceil((maxy - miny) / res_y)
transform = from_origin(minx, maxy, res_x, res_y)

# create output mosaic on disk
profile_base.update({
    'height': height,
    'width': width,
    'transform': transform,
    'dtype': 'float32',
    'count': 1,
    'compress': 'lzw',
    'nodata': np.nan
})

# Split NDVI files into batches to avoid memory issues
ndvi_batches = [ndvi_paths[i:i + batch_size] for i in range(0, len(ndvi_paths), batch_size)]

with rasterio.open(output_path_ndvi, "w+", **profile_base) as dst: # open the resultant tiff
    for batch_idx, batch in enumerate(ndvi_batches, 1):
        print(f"Processing batch {batch_idx}/{len(ndvi_batches)}...")
        for ndvi_file in tqdm(batch, desc=f"Batch {batch_idx}"):
            with rasterio.open(ndvi_file) as src:
                data = src.read(1).astype(np.float32)
                rows, cols = data.shape

                # compute offsets relative to mosaic origin
                col_off = int(round((src.bounds.left - minx) / res_x))
                row_off = int(round((maxy - src.bounds.top) / res_y))

                # clip window to mosaic bounds
                win_col_start = max(col_off, 0)
                win_row_start = max(row_off, 0)
                win_col_end = min(col_off + cols, width)
                win_row_end = min(row_off + rows, height)

                # skip tiles completely outside mosaic
                if win_col_start >= win_col_end or win_row_start >= win_row_end:
                    continue

                # corresponding slice of NDVI tile
                data_row_start = max(0, 0 - row_off)
                data_col_start = max(0, 0 - col_off)
                data_row_end = data_row_start + (win_row_end - win_row_start)
                data_col_end = data_col_start + (win_col_end - win_col_start)
                data_slice = (slice(data_row_start, data_row_end), slice(data_col_start, data_col_end))

                # read mosaic block
                mosaic_block = dst.read(1, window=rasterio.windows.Window(
                    win_col_start, win_row_start,
                    win_col_end - win_col_start, win_row_end - win_row_start),
                    boundless=True, fill_value=np.nan
                )

                # fill only where mosaic is nodata
                tile_block = data[data_slice]
                fill_mask = np.isnan(mosaic_block) & (~np.isnan(tile_block))
                mosaic_block[fill_mask] = tile_block[fill_mask]

                # write back updated block
                dst.write(mosaic_block, 1, window=rasterio.windows.Window(
                    win_col_start, win_row_start,
                    win_col_end - win_col_start, win_row_end - win_row_start)
                )

print(f"NDVI mosaic saved at {output_path_ndvi}")

In [ ]:
# Print a window to check the outputs
with rasterio.open(output_path_ndvi) as src:
    window = rasterio.windows.Window(col_off=15000, row_off=9000, width=7000, height=7000)  
    data = src.read(1, window=window)    
    transform = src.window_transform(window) # get transform for plotting

fig, ax = plt.subplots(figsize=(10,10))
img = ax.imshow(data, cmap='YlGn', vmin=0, vmax=1)
ax.set_title("NDVI Mosaic (sample window)")
plt.colorbar(img, ax=ax, label="NDVI")
plt.show()
